# KGW watermark robustness

Compare identity, comment removal, reformatting, docstring removal, and local-variable renaming on saved KGW generations. Both clean and watermarked solutions are attacked at strengths 1, 2, and 3.

The default source is the local GPU experiment from `watermark_comparison.ipynb`. Change `RUN_ROOT` for an imported or CPU experiment. Set `RUN_SCORING=True` and `RUN_CORRECTNESS=True` to measure detection and correctness; with the defaults, only preparation and a pending report run. Correctness requires local Docker. Results are saved separately under each KGW run's `robustness` folder and displayed below.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys
from datetime import datetime, timezone
from IPython.display import Image, Markdown, display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts/robustness.py').is_file())
RUN_ROOT = ROOT / 'runs/gpu/humaneval/watermark_comparison'
RUN_DIRS = [RUN_ROOT / f'delta-{delta}' for delta in (1, 2, 3)]
PYTHON = next((str(p) for p in (ROOT / '.venv-colab/bin/python', ROOT / '.venv-gpu/Scripts/python.exe', ROOT / '.venv/Scripts/python.exe') if p.is_file()), sys.executable)
RUN_SCORING = False
RUN_CORRECTNESS = False
SCORE_OPTIONS = []  # For deliberate CPU scoring: ['--device', 'cpu', '--dtype', 'float32']
for directory in RUN_DIRS:
    if not (directory / 'manifest.json').is_file():
        raise FileNotFoundError(f'Point RUN_ROOT to your saved experiment: {directory}')
    manifest = json.loads((directory / 'manifest.json').read_text(encoding='utf-8'))
    algorithm = manifest['config']['watermark'].get('algorithm', 'KGW')
    if algorithm != 'KGW':
        raise ValueError(f'Expected a KGW run, found {algorithm}: {directory}')
print('Interpreter:', PYTHON)
print('Source runs:', RUN_ROOT)


In [ ]:
LOG_DIR = ROOT / '.tmp/kgw-robustness-logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)

def run(action, directory, *options):
    log = LOG_DIR / f"{directory.name}-{action}-{datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-%f')}.log"
    args = [PYTHON, '-u', 'scripts/robustness.py', action, '--run-dir', str(directory), *options]
    print(f'{action}: {directory.name}; log: {log}', flush=True)
    with log.open('w', encoding='utf-8') as handle:
        with subprocess.Popen(args, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, encoding='utf-8', errors='replace',
                              env={**os.environ, 'MPLBACKEND': 'Agg'}) as process:
            try:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    handle.write(line)
                    handle.flush()
                status = process.wait()
            except BaseException:
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
                raise
    if status:
        raise RuntimeError(f'{action} failed (exit {status}); full log: {log}')


## 1. Prepare frozen variants

Identity, comment removal, AST reformatting, docstring removal, and conservative local-variable renaming. AST edits also normalize formatting, so use the reformat arm as a control. No-op edits and failures remain in the report. Source code is parsed here, never executed.


In [ ]:
for directory in RUN_DIRS:
    run('prepare', directory)


## 2. Rescore identity and attacks

Set `RUN_SCORING=True` above to run. KGW scoring uses the tokenizer and watermark detector; it does not need SWEET entropy calculations. All variants, including identity, use the same detector environment; old generation scores are not substituted. To run this stage in Colab, transfer the project and full source run including the frozen `robustness` folders, select its project environment, and adjust `RUN_ROOT`. Copy the scored folders back for local Docker evaluation. Completed rows are reused on resume.

When transferring frozen variants to another Python version, skip the preparation cell and start with scoring. Preparation checks the original Python version.


In [ ]:
if RUN_SCORING:
    for directory in RUN_DIRS:
        run('score', directory, *SCORE_OPTIONS)
else:
    print('Scoring not requested. Existing saved scores can still be analyzed.')


## 3. Test correctness in Docker

Start Docker Desktop and set `RUN_CORRECTNESS=True` above. The evaluator validates canonical reference solutions first, then tests the frozen variants with the source run's limits. Correctness alone does not establish evasion; detection scores are also needed.


In [ ]:
if RUN_CORRECTNESS:
    for directory in RUN_DIRS:
        run('evaluate', directory)
else:
    print('Correctness evaluation not requested. Missing results remain pending.')


## 4. Robustness results

A correctness-preserving evasion needs a baseline that passed and was detected, a changed candidate that still passes, and a valid attacked score below or equal to the unchanged threshold. Unscored code is not a successful evasion. No eligible baselines makes the evasion rate unavailable. These fixed attacks do not measure adaptive rewriting or prove semantic equivalence.


In [ ]:
for directory in RUN_DIRS:
    run('analyze', directory)
    output = directory / 'robustness'
    summary = json.loads((output / 'robustness_summary.json').read_text(encoding='utf-8'))
    display(Markdown(f"### KGW {directory.name}: {summary['status']}"))
    text = (output / 'report.md').read_text(encoding='utf-8')
    display(Markdown(text.replace('![Detection and correctness after transformations](robustness.png)', '')))
    if (output / 'robustness.png').is_file():
        display(Image(filename=str(output / 'robustness.png')))
